<a href="https://colab.research.google.com/github/parshav42/50_ML_models/blob/main/labelsugrcane.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch


In [ ]:
!unzip -b /content/project-7-at-2026-08-27-12-59-44e28600.zip -d /content/sugarcaneimge

In [ ]:
!pip install ultralytics

In [ ]:
import os
import random
import shutil

image_folder = "/content/sugarcaneimge/images"
label_folder = "/content/sugarcaneimge/labels"
output_folder = "/content/dataset"

random.seed(42)

images = [
    f for f in os.listdir(image_folder)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

random.shuffle(images)

split = int(len(images) * 0.8)
train_images = images[:split]
val_images = images[split:]

for folder in [
    "images/train",
    "images/val",
    "labels/train",
    "labels/val"
]:
    os.makedirs(os.path.join(output_folder, folder), exist_ok=True)

def copy_files(image_list, split_name):
    for image in image_list:
        name = os.path.splitext(image)[0]

        shutil.copy(
            os.path.join(image_folder, image),
            os.path.join(output_folder, "images", split_name, image)
        )

        label = name + ".txt"
        label_path = os.path.join(label_folder, label)

        if os.path.exists(label_path):
            shutil.copy(
                label_path,
                os.path.join(output_folder, "labels", split_name, label)
            )

copy_files(train_images, "train")
copy_files(val_images, "val")

print("Train:", len(train_images))
print("Validation:", len(val_images))

In [ ]:
import os

label_dirs = [
    "/content/dataset/labels/train",
    "/content/dataset/labels/val"
]

for label_dir in label_dirs:
    if not os.path.exists(label_dir):
        continue

    for filename in os.listdir(label_dir):
        if filename.endswith(".txt"):
            path = os.path.join(label_dir, filename)

            with open(path, "r") as f:
                lines = f.readlines()

            new_lines = []

            for line in lines:
                parts = line.strip().split()

                if parts:
                    old_class = int(parts[0])
                    new_class = old_class - 1
                    parts[0] = str(new_class)

                new_lines.append(" ".join(parts))

            with open(path, "w") as f:
                f.write("\n".join(new_lines) + "\n")

print("Class IDs converted from 1-5 to 0-4")

In [ ]:
!rm -f /content/dataset/labels/train.cache
!rm -f /content/dataset/labels/val.cache

In [ ]:
!yolo detect train model=yolo11n.pt data=/content/dataset/data.yaml epochs=100 imgsz=640 device=0

In [ ]:
!rm rf '/content/sugarcaneimge'

In [ ]:
!rm rf '/content/project-7-at-2026-08-27-12-59-44e28600.zip'